In [1]:
# Test data repair scripts for Chicago

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from phi_data_repair import fill_phi_dates

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_top50_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == "Philadelphia"]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FILLED'] = False

#sub_df_filled = sub_df.copy()
sub_df_filled = fill_phi_dates(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 17.7% -> 17.7%
PERMIT_DATE available (all): 96.1% -> 99.4%
FINAL_DATE available (all): 81.9% -> 82.4%
PERMIT_DATE available (active/final): 99.9% -> 99.9%
FINAL_DATE available (final): 97.5% -> 97.6%


In [4]:
#mask = sub_df_filled["FINAL_DATE"].isna()
mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FILLED']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FILLED']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FILLED']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Final
RECORD_TYPE_ORIGINAL: BP_FIRESUP
FILE_DATE: NaT       *Filled: False*
PERMIT_DATE: 2008-01-18   *Filled: False*
FINAL_DATE: 2008-08-07     *Filled: False*
DATES_DATA: 
{
  "STATUS": "COMPLETED",
  "CONTRACTORZIP": "08066-",
  "MOSTRECENTINSP": "2008/08/07 09:42:00+00",
  "PERMITISSUEDATE": "2008/01/18 10:23:40+00",
  "PERMITCOMPLETEDDATE": "2008/08/07 09:43:03+00",
  "CERTIFICATEOFOCCUPANCYDATE": ""
}


In [5]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "Y": "4862857.28663668",
  "ZIP": "111",
  "\ufeffX": "-8373902.65953111",
  "STATUS": "COMPLETED",
  "ADDRESS": "1575 N 52ND ST",
  "OBJECTID": "171378271",
  "UNIT_NUM": "",
  "GEOCODE_X": "2676296.86671109",
  "GEOCODE_Y": "245243.06292277",
  "OPA_OWNER": "SOUTHEASTERN PENNSYLVANIA",
  "UNIT_TYPE": "",
  "PERMITTYPE": "BP_FIRESUP",
  "TYPEOFWORK": "NEWIN",
  "CENSUSTRACT": "19131-0000",
  "POSSE_JOBID": "",
  "APPLICANTTYPE": "CONTRACTOR",
  "CONTRACTORZIP": "08066-",
  "NUMBEROFUNITS": "",
  "OCCUPANCYTYPE": "",
  "PARCEL_ID_NUM": "543459",
  "USECATEGORIES": "",
  "CONTRACTORCITY": "WEST DEPTFORD",
  "CONTRACTORNAME": "MAJEK FIRE PROTECTION, INC.",
  "MOSTRECENTINSP": "2008/08/07 09:42:00+00",
  "SYSTEMOFRECORD": "HANSEN",
  "ADDRESSOBJECTID": "750709",
  "CONTRACTORSTATE": "NJ",
  "OPA_ACCOUNT_NUM": "882920300",
  "PERMITISSUEDATE": "2008/01/18 10:23:40+00",
  "COUNCIL_DISTRICT": "4",
  "PERMITDESCRIPTION": "SUPPRESSION PERMIT",
  "CONTRACTORADDRESS1": "650 GROVE ROAD"